In [36]:
# Student-Specific Information

from pathlib import Path

LAST_NAME = "Celo"
STUDENT_ID = "TUPM-26-1941"

SEED_DIGIT = int(STUDENT_ID[-1])
target_year = str(2015 + SEED_DIGIT)

print("Target Year:", target_year)

Target Year: 2016


In [37]:
# Workspace

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = WORK_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"

In [38]:
# Total records in dataset
total_records = 0

# Incidents in target year
incidents_in_year = 0

# Total financial loss in target year
total_financial_loss = 0.0

# Count per attack type in target year
attack_counts = {}
incident_records = []

with open(dataset_path, "r", encoding="utf-8") as file:
    header_line = file.readline().strip().split(",")
    
    year_idx = 1
    attack_idx = 2
    loss_idx = 4

    for line in file:
        if not line.strip():
            continue

        total_records += 1
        fields = line.strip().split(",")

        if len(fields) > max(year_idx, attack_idx, loss_idx):
            current_year = fields[year_idx].strip()

            if current_year == target_year:
                incidents_in_year += 1
                
                attack_type = fields[attack_idx].strip()
                attack_counts[attack_type] = attack_counts.get(attack_type, 0) + 1
                
                financial_loss = float(fields[loss_idx].strip())
                total_financial_loss += financial_loss
                
                incident_records.append((financial_loss, attack_type, line))

def sort_key(item):
    return item[0]

# Five Costliest Incidents
sorted_incidents = sorted(incident_records, key=sort_key, reverse=True)
top_5_incidents = sorted_incidents[:5]

avg_financial_loss = total_financial_loss / incidents_in_year if incidents_in_year else 0.0

print("Total records in dataset:", total_records)
print("Incidents in target year:", incidents_in_year)
print("Count per attack type:", attack_counts)
print("Total financial loss (USD millions):", round(total_financial_loss, 3))
print("Average financial loss (USD millions):", round(avg_financial_loss, 3))

print("\nFive Costliest Incidents:")
top_5_pairs = [(item[0], item[1]) for item in top_5_incidents]

printed_count = 0
for loss, attack, line in sorted_incidents:
    fields = line.strip().split(",")
    if len(fields) > max(year_idx, attack_idx, loss_idx):
        financial_loss = float(fields[loss_idx].strip())
        attack_type = fields[attack_idx].strip()
        
        # Matches against the top 5 unique metric records directly
        if (financial_loss, attack_type) in top_5_pairs:
            current_year = fields[year_idx].strip()
            print(f"{'Attack Type:'} {attack_type:<15} {'Financial Loss:'} {financial_loss} Million USD")
            printed_count += 1
            
            if printed_count == 5:
                break


Total records in dataset: 3000
Incidents in target year: 285
Count per attack type: {'DDoS': 53, 'Man-in-the-Middle': 47, 'SQL Injection': 47, 'Phishing': 55, 'Malware': 41, 'Ransomware': 42}
Total financial loss (USD millions): 13947.26
Average financial loss (USD millions): 48.938

Five Costliest Incidents:
Attack Type: Phishing        Financial Loss: 99.98 Million USD
Attack Type: Ransomware      Financial Loss: 99.88 Million USD
Attack Type: Malware         Financial Loss: 99.29 Million USD
Attack Type: Ransomware      Financial Loss: 99.19 Million USD
Attack Type: Phishing        Financial Loss: 98.72 Million USD


In [39]:
# JSON Summary File

import json

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cybersecurity_summary.json"

top_5_json_list = []
for loss, attack, line in top_5_incidents:
    fields = line.strip().split(",")
    top_5_json_list.append({
        "year": fields[year_idx].strip(),
        "attack_type": attack,
        "financial_loss_millions": loss
    })

summary_data = {
    "target_year": int(target_year),
    "total_records_in_dataset": total_records,
    "incidents_in_target_year": incidents_in_year,
    "attack_type_counts": attack_counts,
    "total_financial_loss_millions": round(total_financial_loss, 3),
    "average_financial_loss_millions": round(avg_financial_loss, 3),
    "five_costliest_incidents": top_5_json_list
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

print("JSON Summary File:", json_path)

JSON Summary File: output\Celo_TUPM-26-1941_cybersecurity_summary.json


In [40]:
# Filtered CSV File

filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_filtered_incidents.csv"

with open(filtered_csv_path, "w", encoding="utf-8") as output:
    output.write("Year,Attack Type,Financial Loss (in Million $)\n")
    for loss, attack, line in sorted_incidents:
        output.write(line)

print("Filtered CSV File:", filtered_csv_path)

Filtered CSV File: output\Celo_TUPM-26-1941_filtered_incidents.csv


In [41]:
# Backup Verification

import shutil

backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)

print("Backup verification:", "VERIFIED" if backup_path.exists() else "CHECK")

Backup verification: VERIFIED


In [42]:
# Report File

report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cybersecurity_report.txt"

with open(report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-----------------------------\n")
    report.write(f"Target Year: {target_year}\n")
    report.write(f"Incidents Count: {incidents_in_year}\n")
    report.write(f"Total Financial Loss: {round(total_financial_loss, 3)} Million USD\n")
    report.write(f"Average Financial Loss: {round(avg_financial_loss, 3)} Million USD\n")
    report.write(f"JSON Summary File: {json_path.name}\n")
    report.write(f"Backup File: {backup_path.name}\n")

print("Report file:", report_path)

Report file: output\Celo_TUPM-26-1941_cybersecurity_report.txt


In [43]:
# Final Output
print (f"Student: {LAST_NAME}")
print("Seed:", SEED_DIGIT)

print("\nIncident Counts per Attack Type:")
for attack_type, count in attack_counts.items():
    print(f"{'Attack Type:'} {attack_type:<25} {'Count:'} {count}")

print("\nArchived Findings:")
top_5_pairs = [(item[0], item[1]) for item in top_5_incidents]

printed_count = 0
for loss, attack, line in sorted_incidents:
    fields = line.strip().split(",")
    if len(fields) > max(year_idx, attack_idx, loss_idx):
        financial_loss = float(fields[loss_idx].strip())
        attack_type = fields[attack_idx].strip()
        
        # Matches against the top 5 unique metric records directly
        if (financial_loss, attack_type) in top_5_pairs:
            current_year = fields[year_idx].strip()
            print(f"{'Attack Type:'} {attack_type:<15} {'Financial Loss:'} {financial_loss} Million USD")
            printed_count += 1
            
            if printed_count == 5:
                break

Student: Celo
Seed: 1

Incident Counts per Attack Type:
Attack Type: DDoS                      Count: 53
Attack Type: Man-in-the-Middle         Count: 47
Attack Type: SQL Injection             Count: 47
Attack Type: Phishing                  Count: 55
Attack Type: Malware                   Count: 41
Attack Type: Ransomware                Count: 42

Archived Findings:
Attack Type: Phishing        Financial Loss: 99.98 Million USD
Attack Type: Ransomware      Financial Loss: 99.88 Million USD
Attack Type: Malware         Financial Loss: 99.29 Million USD
Attack Type: Ransomware      Financial Loss: 99.19 Million USD
Attack Type: Phishing        Financial Loss: 98.72 Million USD
